# 01 — Profiling & Suggested DQ Rules (Heuristic AI)

We profile datasets and generate **suggested** rules (as YAML) using heuristics.

> Later, heuristics can be replaced by Azure OpenAI without changing the execution engine.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from pandas.api.types import is_numeric_dtype, is_object_dtype, is_string_dtype

BASE = Path("./data")

## Load data

In [10]:
from pathlib import Path

BASE = Path(r"../data/raw")
mat = pd.read_csv(BASE / "materials_master.csv")
gm = pd.read_csv(BASE / "goods_movements.csv")
mat.shape, gm.shape

((202, 5), (500, 6))

## Simple profiling helpers

In [14]:
def profile_df(df: pd.DataFrame, max_top=10):
    prof = {}
    n = len(df)

    for c in df.columns:
        s = df[c]

        # null / blank logic
        if is_string_dtype(s) or is_object_dtype(s):
            blanks = s.astype("string").str.strip().eq("").sum()
            nulls = int(s.isna().sum() + blanks)
        else:
            nulls = int(s.isna().sum())

        distinct = int(s.nunique(dropna=True))
        top = s.value_counts(dropna=True).head(max_top).to_dict()

        col_prof = {
            "dtype": str(s.dtype),
            "null_count": nulls,
            "null_pct": (nulls / n * 100) if n else 0.0,
            "distinct": distinct,
            "top_values": top,
        }

        # numeric stats
        if is_numeric_dtype(s):
            sn = pd.to_numeric(s, errors="coerce")
            col_prof.update(
                {
                    "min": float(np.nanmin(sn.values)) if sn.notna().any() else None,
                    "max": float(np.nanmax(sn.values)) if sn.notna().any() else None,
                    "mean": float(np.nanmean(sn.values)) if sn.notna().any() else None,
                }
            )

        prof[c] = col_prof

    return {"row_count": n, "columns": prof}


mat_prof = profile_df(mat)
gm_prof = profile_df(gm)

mat_prof["columns"], gm_prof["columns"]

({'material_id': {'dtype': 'str',
   'null_count': 0,
   'null_pct': 0.0,
   'distinct': 200,
   'top_values': {'M00011': 2,
    'M00021': 2,
    'M00001': 1,
    'M00002': 1,
    'M00003': 1,
    'M00004': 1,
    'M00005': 1,
    'M00006': 1,
    'M00007': 1,
    'M00008': 1}},
  'material_name': {'dtype': 'str',
   'null_count': 1,
   'null_pct': 0.49504950495049505,
   'distinct': 200,
   'top_values': {'Material 11': 2,
    'Material 1': 1,
    'Material 2': 1,
    'Material 3': 1,
    'Material 4': 1,
    'Material 5': 1,
    'Material 7': 1,
    'Material 8': 1,
    'Material 9': 1,
    'Material 10': 1}},
  'unit_of_measure': {'dtype': 'str',
   'null_count': 1,
   'null_pct': 0.49504950495049505,
   'distinct': 5,
   'top_values': {'kg': 45, 'm': 44, 'g': 44, 'l': 34, 'pcs': 34}},
  'weight': {'dtype': 'float64',
   'null_count': 0,
   'null_pct': 0.0,
   'distinct': 201,
   'top_values': {37.49: 2,
    83.81: 1,
    40.68: 1,
    50.66: 1,
    58.15: 1,
    34.22: 1,
    50.04

## Suggested rules generator (heuristics)

In [28]:
def suggest_rules_materials(mat_prof):
    # Heuristic assumptions for Materials master
    rules = []
    cols = mat_prof["columns"]

    # schema required columns
    required = [
        "material_id",
        "material_name",
        "unit_of_measure",
        "weight",
        "created_at",
        "ts_load",
    ]
    rules.append(
        {
            "rule_id": "M001_schema_presence",
            "rule_type": "schema",
            "severity": "critical",
            "expectation": {"required_columns": required},
        }
    )

    # PK uniqueness
    rules.append(
        {
            "rule_id": "M002_pk_unique",
            "rule_type": "uniqueness",
            "severity": "critical",
            "expectation": {"column": "material_id", "max_duplicates_allowed": 0},
        }
    )

    # Completeness for key fields
    for c in ["material_id", "material_name", "unit_of_measure"]:
        rules.append(
            {
                "rule_id": f"M003_{c}_not_null",
                "rule_type": "completeness",
                "severity": "critical" if c != "unit_of_measure" else "high",
                "expectation": {"column": c, "max_null_percent": 0},
            }
        )

    # Domain for UOM: infer top values as allowed if they look reasonable
    top_uom = list(cols["unit_of_measure"]["top_values"].keys())
    # naive filter: keep values with length <= 4 and alphabetic
    allowed = [v for v in top_uom if isinstance(v, str) and v.strip() and len(v.strip()) <= 4]
    rules.append(
        {
            "rule_id": "M010_uom_domain",
            "rule_type": "domain",
            "severity": "high",
            "expectation": {"column": "unit_of_measure", "allowed_values": allowed[:10]},
        }
    )

    # Range for weight (if numeric)
    if "min" in cols["weight"] and "max" in cols["weight"]:
        # heuristic: weight should be >=0, and <= max observed * 1.2 (cap)
        max_obs = cols["weight"]["max"]
        rules.append(
            {
                "rule_id": "M011_weight_range",
                "rule_type": "range",
                "severity": "high",
                "expectation": {"column": "weight", "min": 0, "max": round(max_obs * 1.2, 2)},
            }
        )

    # created_at not in future
    rules.append(
        {
            "rule_id": "M012_created_at_not_future",
            "rule_type": "date_not_in_future",
            "severity": "medium",
            "expectation": {"column": "created_at"},
        }
    )

    # data set freshness
    rules.append(
        {
            "rule_id": "M003_dataset_is_fresh",
            "rule_type": "freshness",
            "severity": "critical",
            "expectation": {"column": "ts_load", "max_age_days": 2, "timezone": "UTC"},
        }
    )
    return rules


def suggest_rules_goods_movements(gm_prof):
    rules = []
    required = [
        "movement_id",
        "material_id",
        "quantity",
        "movement_type",
        "plant",
        "movement_date",
        "ts_load",
    ]
    rules.append(
        {
            "rule_id": "G001_schema_presence",
            "rule_type": "schema",
            "severity": "critical",
            "expectation": {"required_columns": required},
        }
    )
    rules.append(
        {
            "rule_id": "G002_movement_id_unique",
            "rule_type": "uniqueness",
            "severity": "critical",
            "expectation": {"column": "movement_id", "max_duplicates_allowed": 0},
        }
    )
    for c in ["movement_id", "material_id", "movement_type", "plant", "movement_date"]:
        rules.append(
            {
                "rule_id": f"G003_{c}_not_null",
                "rule_type": "completeness",
                "severity": "high" if c != "movement_id" else "critical",
                "expectation": {"column": c, "max_null_percent": 0},
            }
        )
    # quantity non-negative
    rules.append(
        {
            "rule_id": "G010_quantity_non_negative",
            "rule_type": "range",
            "severity": "high",
            "expectation": {"column": "quantity", "min": 0, "max": None},
        }
    )
    # movement_type domain inferred
    top_mt = list(gm_prof["columns"]["movement_type"]["top_values"].keys())
    allowed = [v for v in top_mt if isinstance(v, str) and v.strip() and len(v.strip()) <= 12]
    rules.append(
        {
            "rule_id": "G011_movement_type_domain",
            "rule_type": "domain",
            "severity": "high",
            "expectation": {"column": "movement_type", "allowed_values": allowed[:10]},
        }
    )
    # date not in future
    rules.append(
        {
            "rule_id": "G012_movement_date_not_future",
            "rule_type": "date_not_in_future",
            "severity": "medium",
            "expectation": {"column": "movement_date"},
        }
    )

    # data set freshness
    rules.append(
        {
            "rule_id": "G003_dataset_is_fresh",
            "rule_type": "freshness",
            "severity": "critical",
            "expectation": {"column": "ts_load", "max_age_days": 1, "timezone": "UTC"},
        }
    )
    return rules


mat_rules = suggest_rules_materials(mat_prof)
gm_rules = suggest_rules_goods_movements(gm_prof)

len(mat_rules), len(gm_rules)

(9, 11)

## Add a cross-dataset rule: referential integrity (movements.material_id exists in materials.material_id)

In [29]:
ref_rule = {
    "rule_id": "X001_material_fk_exists",
    "rule_type": "referential_integrity",
    "severity": "critical",
    "expectation": {
        "child_dataset": "goods_movements",
        "child_column": "material_id",
        "parent_dataset": "materials_master",
        "parent_column": "material_id",
    },
}

## Export suggested rules to YAML files (rules-as-code)

In [30]:
out_dir = Path(r"../dq_registry") / "rulesets"
out_dir.mkdir(exist_ok=True)


def export_ruleset(
    dataset_id, rules, version=1, owner_team="SupplyChain", data_owner="owner@example.com"
):
    doc = {
        "dataset_id": dataset_id,
        "ruleset_version": version,
        "owner_team": owner_team,
        "data_owner": data_owner,
        "rules": rules,
    }
    p = out_dir / f"{dataset_id}.yaml"
    p.write_text(yaml.safe_dump(doc, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return p


p1 = export_ruleset("materials_master", mat_rules, version=1)
p2 = export_ruleset("goods_movements", gm_rules, version=1)
p3 = export_ruleset("cross_dataset", [ref_rule], version=1)
print("Wrote:", p1, p2, p3, sep="\n")

Wrote:
..\dq_registry\rulesets\materials_master.yaml
..\dq_registry\rulesets\goods_movements.yaml
..\dq_registry\rulesets\cross_dataset.yaml


Open the YAML files to review/edit rules before execution.